In [6]:
import pandas as pd

def build_seed_and_export_csv_ppmi():
    print("==================================================")
    print("🕵️ 步骤 1: 加载 PrimeKG 数据库...")
    kg_df = pd.read_csv('kg.csv', low_memory=False)
    
    # 提取所有原生节点（保持原始大小写，用于精准匹配基因等）
    all_nodes = set(kg_df['x_name'].dropna().unique()).union(
        set(kg_df['y_name'].dropna().unique())
    )
    # 提取小写节点集合（用于临床表型的模糊容错匹配）
    kg_nodes_lower = set(kg_df['x_name'].dropna().str.lower().unique()).union(
        set(kg_df['y_name'].dropna().str.lower().unique())
    )
    print(f"   ✅ 成功加载 kg.csv，共提取独立节点 {len(all_nodes)} 个。")

    print("\n==================================================")
    print("🧬 步骤 2: 初始化医学共识核心种子 (Core Seeds)...")
    
    # 新增：直接引入外部医学先验知识（疾病与帕金森核心基因）
    core_seeds = {
        "Parkinson disease": "Core_Disease",
        "LRRK2": "Core_Gene",
        "GBA": "Core_Gene",
        "SNCA": "Core_Gene"
    }

    print("\n==================================================")
    print("📊 步骤 3: 构建 PPMI 临床表型映射字典...")
    
    ppmi_phenotypes = {
        # --- 认知 (保留重要的临床辅助诊断量表) ---
        "moca_score": "Memory impairment",
        "mmse_score": "Memory impairment",
        "cdr_score": "Dementia",

        # --- 中轴症状与面部 ---
        "code_upd2301_speech_problems": "speech disorder",
        "code_upd2302_facial_expression": "bradykinesia",
        "code_upd2309_arising_from_chair": "bradykinesia",
        "code_upd2310_gait": "Unsteady gait",
        "code_upd2311_freezing_of_gait": "freezing of gait",
        "code_upd2312_postural_stability": "Unsteady gait",
        "code_upd2313_posture": "Unsteady gait",
        "code_upd2314_body_bradykinesia": "bradykinesia",
        
        # --- 强直 (细化到具体部位，捕获颈部特异性) ---
        "code_upd2303a": "Nuchal rigidity", # 颈部强直
        "code_upd2303b": "Rigidity",        # 右上肢
        "code_upd2303c": "Rigidity",        # 左上肢
        "code_upd2303d": "Rigidity",        # 右下肢
        "code_upd2303e": "Rigidity",        # 左下肢
        
        # --- 运动迟缓/灵活性 (拆分左右侧，保留疾病起病与发展的单双侧不对称性) ---
        "code_upd2304a": "bradykinesia", # 右手敲击
        "code_upd2304b": "bradykinesia", # 左手敲击
        "code_upd2305a": "bradykinesia", # 右手运动
        "code_upd2305b": "bradykinesia", # 左手运动
        "code_upd2306a": "pronation-supination of the forearm, impairment of", # 右手旋前旋后
        "code_upd2306b": "pronation-supination of the forearm, impairment of", # 左手旋前旋后
        "code_upd2307a": "bradykinesia", # 右脚趾
        "code_upd2307b": "bradykinesia", # 左脚趾
        "code_upd2308a": "bradykinesia", # 右下肢敏捷
        "code_upd2308b": "bradykinesia", # 左下肢敏捷
        
        # --- 震颤 (使用精细化震颤节点) ---
        "code_upd2315a": "Action tremor",  # 右手姿势性震颤
        "code_upd2315b": "Action tremor",  # 左手姿势性震颤
        "code_upd2316a": "Kinetic tremor", # 右手动作性震颤
        "code_upd2316b": "Kinetic tremor", # 左手动作性震颤
        "code_upd2317a": "resting tremor", # 面部静止性
        "code_upd2317b": "resting tremor", # 右上肢
        "code_upd2317c": "resting tremor", # 左上肢
        "code_upd2317d": "resting tremor", # 右下肢
        "code_upd2317e": "resting tremor", # 左下肢
        "code_upd2318_consistency_of_rest_tremor": "resting tremor",

        # --- 运动并发症 ---
        "upd2da_dyskinesias_during_exam": "Dyskinesia",
        "upd2db_movements_interfere_with_ratings": "Dyskinesia"
    }

    valid_records = []
    print("\n==================================================")
    print("🔍 步骤 4: 在 PrimeKG 中校验节点有效性...")
    

    # LRRK2、GBA 和 SNCA 是帕金森病的核心基因，直接作为核心先验知识种子进行校验
    # 1. 优先校验核心先验知识 (疾病与基因)
    for seed, category in core_seeds.items():
        if seed in all_nodes:
            valid_records.append({"PPMI_Feature": "Core_Prior_Knowledge", "PrimeKG_Node": seed, "Category": category})
            print(f"     ✅ 命中: {seed}")
        else:
            print(f"     ❌ 未命中: {seed}")

    # 2. 校验临床表型映射
    for feat, node_name in ppmi_phenotypes.items():
        if node_name.lower() in kg_nodes_lower:
            # 尝试获取原始正确的大写节点名以保持图谱规范
            actual_node = next((n for n in all_nodes if n.lower() == node_name.lower()), node_name)
            print(f"     ✅ 命中: {actual_node} (对应 {feat})")
            valid_records.append({"PPMI_Feature": feat, "PrimeKG_Node": actual_node, "Category": "Phenotype"})
        else:
            print(f"     ❌ 未命中: {node_name} (对应 {feat})")

    print("\n==================================================")
    print("🚫 步骤 5: 附加全局黑名单规则...")
    for rule in ["drug", "exposure", "indication", "contraindication", "off-label use"]:
        cat = "BLACKLIST_TYPE" if rule in ["drug", "exposure"] else "BLACKLIST_REL"
        valid_records.append({"PPMI_Feature": cat, "PrimeKG_Node": rule, "Category": "Rule"})

    OUTPUT_CSV_PATH = "PPMI_Seed_DementiaHKG.csv"
    print("\n==================================================")
    print("💾 步骤 6: 导出为 CSV 文件...")
    out_df = pd.DataFrame(valid_records)
    out_df.to_csv(OUTPUT_CSV_PATH, index=False, encoding='utf-8')
    print(f"   ✅ 导出完成！文件已保存至: {OUTPUT_CSV_PATH}")

if __name__ == "__main__":
    build_seed_and_export_csv_ppmi()

🕵️ 步骤 1: 加载 PrimeKG 数据库...
   ✅ 成功加载 kg.csv，共提取独立节点 129262 个。

🧬 步骤 2: 初始化医学共识核心种子 (Core Seeds)...

📊 步骤 3: 构建 PPMI 临床表型映射字典...

🔍 步骤 4: 在 PrimeKG 中校验节点有效性...
     ✅ 命中: Parkinson disease
     ✅ 命中: LRRK2
     ✅ 命中: GBA
     ✅ 命中: SNCA
     ✅ 命中: Memory impairment (对应 moca_score)
     ✅ 命中: Memory impairment (对应 mmse_score)
     ✅ 命中: Dementia (对应 cdr_score)
     ✅ 命中: speech disorder (对应 code_upd2301_speech_problems)
     ✅ 命中: Bradykinesia (对应 code_upd2302_facial_expression)
     ✅ 命中: Bradykinesia (对应 code_upd2309_arising_from_chair)
     ✅ 命中: Unsteady gait (对应 code_upd2310_gait)
     ✅ 命中: Freezing of gait (对应 code_upd2311_freezing_of_gait)
     ✅ 命中: Unsteady gait (对应 code_upd2312_postural_stability)
     ✅ 命中: Unsteady gait (对应 code_upd2313_posture)
     ✅ 命中: Bradykinesia (对应 code_upd2314_body_bradykinesia)
     ✅ 命中: Nuchal rigidity (对应 code_upd2303a)
     ✅ 命中: Rigidity (对应 code_upd2303b)
     ✅ 命中: Rigidity (对应 code_upd2303c)
     ✅ 命中: Rigidity (对应 code_upd2303d)
     ✅ 命中: